# 02B Join Validation And Final Data Model

This notebook documents the join strategy used for Olist and validates that the final master table keeps one row per order for downstream analysis and Tableau.


In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from scripts.etl_pipeline import build_clean_tables, write_outputs


In [2]:
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

_, clean_tables = build_clean_tables(RAW_DIR)
exports = write_outputs(PROCESSED_DIR, clean_tables)
orders_master = exports['olist_orders_master.csv']
items_enriched = exports['olist_order_items_enriched.csv']


In [3]:
# Final data model shape.
model_summary = pd.DataFrame([
    {'dataset': 'olist_orders_master.csv', 'granularity': 'one row per order', 'rows': len(orders_master), 'columns': len(orders_master.columns)},
    {'dataset': 'olist_order_items_enriched.csv', 'granularity': 'one row per order item', 'rows': len(items_enriched), 'columns': len(items_enriched.columns)},
])
model_summary


,dataset,granularity,rows,columns
0,olist_orders_master.csv,one row per order,99441,50
1,olist_order_items_enriched.csv,one row per order item,112650,26


In [4]:
# Validate order-level uniqueness.
{
    'rows_in_orders_master': len(orders_master),
    'unique_order_id_in_orders_master': int(orders_master['order_id'].nunique()),
    'duplicate_order_ids_in_orders_master': int(orders_master['order_id'].duplicated().sum()),
}


{'rows_in_orders_master': 99441,
 'unique_order_id_in_orders_master': 99441,
 'duplicate_order_ids_in_orders_master': 0}

In [5]:
# Check important downstream fields.
important_fields = [
    'order_id', 'order_status', 'customer_state', 'primary_product_category',
    'primary_seller_state', 'order_value', 'payment_type_primary',
    'review_score', 'delivery_days', 'estimated_delivery_gap_days', 'is_late_delivery'
]
orders_master[important_fields].head(10)


,order_id,order_status,customer_state,primary_product_category,primary_seller_state,order_value,payment_type_primary,review_score,delivery_days,estimated_delivery_gap_days,is_late_delivery
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,SP,housewares,SP,38.71,voucher,4.0,8.436574,-7.107488,0
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,BA,perfumery,SP,141.46,boleto,4.0,13.782037,-5.355729,0
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,GO,auto,SP,179.12,credit_card,5.0,9.394213,-17.245498,0
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,RN,pet_shop,MG,72.20,credit_card,5.0,13.208750,-12.980069,0
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,SP,stationery,SP,28.62,credit_card,5.0,2.873877,-9.238171,0
5,a4591c265e18cb1dcee52889e2d8acc3,delivered,PR,auto,SP,175.26,credit_card,4.0,16.542245,-5.543113,0
6,136cce7faa42fdb2cefd53fdc79a6098,invoiced,RS,unknown,SP,65.95,credit_card,2.0,NaN,NaN,0
7,6514b8ad8028c9f2cc2374ded245783f,delivered,RJ,auto,SP,75.16,credit_card,5.0,9.989826,-11.461215,0
8,76c6e866289321a7c93b82b54852dc33,delivered,RS,furniture_decor,SP,35.95,boleto,1.0,9.818762,-31.410995,0
9,e69bfb5eb88e0ed6a785585b27e16dbf,delivered,SP,office_furniture,SP,169.76,credit_card,5.0,18.221852,-6.281597,0


In [6]:
# Null diagnostic for the fields most likely to matter in Tableau and analysis.
orders_master[important_fields].isna().sum().sort_values(ascending=False)


delivery_days                  2965
estimated_delivery_gap_days    2965
primary_product_category        775
primary_seller_state            775
payment_type_primary              1
order_id                          0
order_status                      0
customer_state                    0
order_value                       0
review_score                      0
is_late_delivery                  0
dtype: int64

## Join Strategy Used

1. Clean each raw table separately.
2. Aggregate geolocation to one zip-prefix lookup.
3. Enrich customers and sellers with zip-based geography.
4. Enrich order items with product and seller details.
5. Aggregate order items, payments, and reviews to the order level.
6. Join the aggregated tables back into `orders` to create one master order dataset.

This keeps Tableau simple and prevents duplicate measures caused by raw one-to-many joins.
